# 05. Policy And Sidecars

This notebook shows two important v1 ideas together:

- policy context narrows what a caller may see
- artifact and vector sidecars stay subordinate to semantic control

We will register one protected artifact, register one vector record anchored to the journal, and then query it with and without the right capability.


In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root


In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_http_service, stop_http_service
from aether_sdk import (
    AetherApiError,
    AetherClient,
    make_artifact_reference,
    make_datom,
    make_policy,
    make_policy_context,
    make_vector_record,
    value_string,
)

ensure_rust_toolchain()
session = start_http_service(repo_root)
client = AetherClient(session.base_url)

pretty_json(client.health())


## Anchor The Sidecar Records To The Journal

Sidecar registrations are tied to the journal tail, so we create two anchor elements first.


In [ ]:
client.append(
    [
        make_datom(entity=1, attribute=1, value=value_string("sidecar-anchor-1"), element=1, op="Annotate"),
        make_datom(entity=1, attribute=1, value=value_string("sidecar-anchor-2"), element=2, op="Annotate"),
    ]
)

pretty_json(client.history())


In [ ]:
client.register_artifact_reference(
    make_artifact_reference(
        sidecar_id="semantic-memory",
        artifact_id="doc-1",
        entity=41,
        uri="s3://aether/docs/doc-1.md",
        media_type="text/markdown",
        byte_length=256,
        digest="sha256:doc-1",
        metadata={"kind": {"String": "runbook"}},
        registered_at=1,
        policy=make_policy(capability="memory_reader"),
    )
)

client.register_vector_record(
    record=make_vector_record(
        sidecar_id="semantic-memory",
        vector_id="vec-1",
        entity=41,
        source_artifact_id="doc-1",
        embedding_ref="s3://aether/vectors/vec-1.bin",
        dimensions=3,
        metric="cosine",
        metadata={"topic": {"String": "handoff"}},
        registered_at=2,
        policy=make_policy(capability="memory_reader"),
    ),
    embedding=[0.9, 0.1, 0.0],
)


In [ ]:
try:
    client.get_artifact_reference(sidecar_id="semantic-memory", artifact_id="doc-1")
except AetherApiError as exc:
    print(exc)


In [ ]:
artifact = client.get_artifact_reference(
    sidecar_id="semantic-memory",
    artifact_id="doc-1",
    policy_context=make_policy_context(capabilities=["memory_reader"]),
)
pretty_json(artifact)


In [ ]:
search = client.search_vectors(
    {
        "sidecar_id": "semantic-memory",
        "query_embedding": [1.0, 0.0, 0.0],
        "top_k": 1,
        "metric": "cosine",
        "as_of": 2,
        "projection": {
            "predicate": {
                "id": 81,
                "name": "semantic_neighbor",
                "arity": 3,
            },
            "query_entity": 999,
        },
        "policy_context": make_policy_context(capabilities=["memory_reader"]),
    }
)
pretty_json(search)


## What This Proves

- sidecars do not float free from semantic time
- policy context can narrow access cleanly
- vector matches can project back into semantic facts with provenance

That is the current v1 posture: useful memory surfaces without surrendering semantic control.


In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)
